# CORAK — Simulation Cases

Companion notebook for the CORAK paper, Sections 5.2–5.3 and Appendices C.1–C.2.

This notebook contains two simulation cases:
- **Case 1:** Multi-morbidity extension (Thiem et al. 2022) → Section 5.2 / Appendix C.1
- **Case 2:** Swiss Minaret Vote (Baumgartner & Epple 2014) → Section 5.3.1 / Appendix C.2

Scripts: `simulation_study.py` (Case 1), `minaret_simulation_study.py` (Case 2)  
Data: `multimorbidity_data.csv`, `minaret_data.csv`


## Setup

Install CORA and CORAK if running in Colab or a fresh environment.

In [ ]:
# Uncomment if running in Google Colab or without a pre-configured environment
# !pip install git+https://github.com/PoliUniLu/cora.git -q
# !pip install git+https://github.com/<your-org>/corak.git -q

import sys, warnings
warnings.filterwarnings("ignore")

# For local development: ensure the corak package is on the path
# sys.path.insert(0, "../..")

import pandas as pd
import cora as cora_pkg
from corak import CorakContext, UNDEF

---

## Case 1: Multi-Morbidity Extension
### Appendix C.1 / Section 5.2

**Dataset:** Thiem et al. (2022), 10-row truth table, N=40 synthetic cases.  
**Ground truth:** DEP = D·H (diabetes × hypertension → depression onset).  
**Scripts:** `simulation_study.py`, data in `multimorbidity_data.csv`.  
**Purpose:** Direct extension of the CORA showcase example; demonstrates CORAK on the dataset where CORA was originally introduced.


## 1. Dataset

The dataset is a structured extension of Thiem et al. (2022)'s multi-morbidity example.
Five conditions (D, H, K, L, C) and two outcomes (DEP, CVD).
⊥ values are encoded as -1.

| Code | Condition | ⊥ = |
|------|-----------|------|
| D | Type 2 Diabetes | Glucose test not performed |
| H | Hypertension | BP measurement not taken |
| K | Chronic Kidney Disease | eGFR not assessed |
| L | Depressive episode | PHQ-9 not administered |
| C | Cardiovascular event history | ECG/troponin not assessed |

In [ ]:
# Load dataset (⊥ encoded as -1)
df = pd.read_csv("multimorbidity_data.csv", index_col="case")
df = df[["D","H","K","L","C","DEP","CVD"]]

print("Dataset (−1 = structurally indefinite ⊥):")
print(df.to_string())
print(f"
Rows: {len(df)} | Outcomes: DEP, CVD")

## 2. Row Classification

CORAK classifies each row as:
- **C1**: Fully defined — no ⊥ in any column
- **C2**: Condition-indefinite — at least one ⊥ in condition columns
- **C3**: Outcome-indefinite — ⊥ in outcome columns, all conditions defined

In [ ]:
ctx = CorakContext(
    df,
    output_labels=["DEP", "CVD"],
    undef_value=-1,
    algorithm="ON-OFF"
)

print(ctx)
print()

classif = ctx.get_row_classification()
print("Row classification:")
print(classif[["row_class","undef_conditions","undef_outcomes"]].to_string())
print(f"
C1: {ctx.n_c1} rows | C2: {ctx.n_c2} rows | C3: {ctx.n_c3} rows")

## 3. Tier 1 Solutions

CORAK Tier 1 runs CORA exclusively on C1 rows (Cases 1, 2, 4, 5, 7, 8, 10).
By Lemma 1, this is identical to CORA when no ⊥ values are present.

In [ ]:
print("=== CORAK Tier 1 — Irredundant Systems ===")
for s in ctx.get_irredundant_systems():
    print(s)

## 4. Kleene Consistency Intervals

For each Tier 1 prime implicant, CORAK computes:
- **Cons_def**: CORA's point estimate (C1 rows only) 
- **Cons_lower**: worst case (⊥ outcomes → 0)
- **Cons_upper**: best case (⊥ outcomes → 1)
- **audit_width**: Cons_upper − Cons_lower (0 = robust, >0 = sensitive)

In [ ]:
intervals = ctx.get_consistency_intervals()
print("Kleene Consistency Intervals (Proposition 1):")
print(intervals.to_string(index=False))

## 5. Tier 2 — Conditional INUS Structures

For each C2 row, CORAK enumerates resolutions of ⊥-valued conditions
and reports conditional INUS structures — valid if the ⊥ condition
resolves to a specific value.

In [ ]:
pi_cond = ctx.get_conditional_prime_implicants()
if pi_cond:
    print(f"Found {len(pi_cond)} conditional INUS structure(s):")
    for pi in pi_cond:
        print(f"  {pi}")
else:
    print("No conditional structures above threshold.")

## 6. CORA vs. CORAK Comparison

CORA runs on all rows with ⊥ silently coded as 0.
CORAK Tier 1 runs on C1 rows only.

In [ ]:
# CORA: all ⊥ coded as 0
df_cora = df.replace(-1, 0)
active = [c for c in ["D","H","K","L","C"] if df_cora[c].nunique() > 1]
ctx_cora = cora_pkg.OptimizationContext(
    df_cora, ["DEP","CVD"], input_labels=active, algorithm="ON-OFF"
)

print("=== CORA (⊥→0, all rows) ===")
for s in ctx_cora.get_irredundant_systems():
    print(s)

print()
print("=== CORAK Tier 1 (C1 rows only) ===")
for s in ctx.get_irredundant_systems():
    print(s)

print()
print("Consistency comparison:")
cora_dets = ctx_cora.pi_details()[["PI","Inc."]].rename(columns={"Inc.":"CORA_Cons"})
corak_dets = ctx.get_consistency_intervals()[["PI","Cons_def","Cons_lower","Cons_upper","audit_width"]]
try:
    merged = cora_dets.merge(corak_dets, on="PI", how="outer")
    print(merged.to_string(index=False))
except Exception:
    print("CORA PIs:"); print(cora_dets.to_string(index=False))
    print("CORAK intervals:"); print(corak_dets.to_string(index=False))

## 7. Sensitivity Analysis

The sensitivity table varies α — the fraction of ⊥ values recoded as 0.
- α = 0.0 → CORAK Tier 1 (no recoding)
- α = 1.0 → standard CORA (all ⊥ → 0)

Stability across the row indicates robustness to the ⊥ classification.
Divergence flags where the distinction matters.

In [ ]:
sens = ctx.sensitivity_table(alphas=[0.0, 0.25, 0.5, 0.75, 1.0])
print("Sensitivity Table (α = fraction of ⊥ recoded as 0):")
print(sens.to_string(index=False))
print()
print("α = 0.0 → CORAK Tier 1 | α = 1.0 → CORA")
print("Stable solutions across α → CORA was appropriate.")
print("Divergence → ⊥ classification matters for conclusions.")

## 8. Key Findings

| Metric | CORA | CORAK Tier 1 | CORAK Tier 2 |
|--------|------|--------------|---|
| Undefined cases signalled | None | None explicit | Cases 3, 6, 9 with types |
| Consistency scores | Inflated (includes ⊥ rows) | Certified (C1 only) | Interval [lo, hi] |
| Case 9 (H=⊥) impact | Ignored | Excluded from ON-set | Surfaced as conditional D·¬H→DEP |
| Case 6 (DEP=⊥) impact | Coded as OFF (DEP=0) | Excluded from OFF-set | Not used for cube extension |

**Clinical interpretation:** A screening protocol derived from CORA would not flag T2D patients without measured hypertension for depression monitoring. CORAK signals that this exclusion rests on an unverified assumption — Case 9 shows that if H=0 is the correct resolution, then D·¬H is a sufficient pathway that CORA misses.

---

## References

- Thiem, A., Mkrtchyan, L., & Sebechlebská, Z. (2022). Combinational Regularity Analysis (CORA). *BMC Medical Research Methodology*, 22(1), 333.
- Sebechlebská, Z., Mkrtchyan, L., & Thiem, A. (2023). CORA and LOGIGRAM. *JOSS*, 8(85), 5019.
- CORAK paper: *Combinational Regularity Analysis with Kleene-Valued Conditions* (2026, working paper).

**Reproduce with:**


---

## Case 2: Swiss Minaret Vote
### Appendix C.2 / Section 5.3.1

**Dataset:** Baumgartner & Epple (2014), N=26 Swiss cantons, `d.minaret` in the `cna` R package.  
**Ground truth:** X → M (new xenophobia → minaret ban), consistency=1.00, coverage=0.86.  
**Scripts:** `minaret_simulation_study.py`, data in `minaret_data.csv`.  
**Purpose:** Independent replication on a non-CORA-group dataset with a different causal structure (two prime implicants: X and T).


---

## Section 5.4 — Replication: Swiss Minaret Vote (Baumgartner & Epple 2014)

This section reproduces the simulation of CORAK Section 5.4. It uses the Swiss Minaret Vote dataset (`d.minaret` in the `cna` R package) — an independent dataset from a different research team and domain — to test whether CORAK's advantages over CORA generalise beyond the multi-morbidity example.

**Variables:**
- A: prior anti-Islamic voting sentiment
- L: native foreign-language speaker share  
- S: traditional economic sector
- T: traditional political structures
- X: new xenophobia (intermediate variable, key driver of M)
- M: canton voted for the 2009 minaret ban (outcome)

**Ground truth (Baumgartner & Epple 2014):** X → M at consistency=1.00, coverage=0.86. T provides an alternative pathway for FR, VS, JU.


### Methodological context: Thiem's critique of Baumgartner & Epple (2014)

The Minaret dataset was chosen as a second benchmark because it comes from an independent research team. However, this independence must be qualified: Thiem (2015, *SMR* 44(4)) has argued that Baumgartner & Epple selected this dataset specifically to demonstrate a weakness of the **Quine-McCluskey (QMC) algorithm** — the "one-difference restriction" — not a weakness of QCA or CCMs in general. Thiem shows that the eQMC algorithm (ON-OFF, using both positive and negative minterms) recovers the same chain structure that CNA finds.

**Two implications for CORAK:**

1. The target ground truth **X→M is not contested**: both B&E's CNA and Thiem's QCA confirm X as a sufficient condition for M. The recovery metric is agreed upon by all parties.

2. **CORA uses ON-OFF** (McCluskey 1965) — the same algorithmic logic that Thiem vindicated. The B&E critique of QMC does not apply to CORA or CORAK. The dataset is therefore not adversarial to CORA in the way it was meant to be adversarial to QMC-based QCA.

**Reference:** Thiem, A. (2015). Using QCA for identifying causal chains: A commentary on Baumgartner and Epple (2014). *Sociological Methods & Research*, 44(4), 723–736.


In [ ]:
import pandas as pd
import numpy as np
import sys

# Minaret dataset (N=26 Swiss cantons)
CANTONS = [
    'LU','UR','SZ','OW','NW','AR','AI','GL','ZG','SO','SG','AG',
    'VD','NE','GE','GR','TG','ZH','BE','FR','BS','BL','SH','TI','VS','JU'
]
minaret_data = {
    'A': [1,1,1,1,1,1,1,1,1,1,1,1, 0,0,0,0,0,1,1,1,1,1,0,0,0,0],
    'L': [0,0,0,0,0,0,0,0,0,0,0,0, 1,1,1,0,0,1,1,0,1,1,1,0,0,1],
    'S': [1,1,1,1,1,1,1,1,1,1,1,1, 0,0,0,1,1,1,1,0,0,0,1,0,0,0],
    'T': [1,1,1,1,1,1,1,1,1,1,1,1, 0,0,0,1,1,0,1,1,0,0,0,0,1,1],
    'X': [1,1,1,1,1,1,1,1,1,1,1,1, 0,0,0,1,1,1,1,0,0,1,1,1,0,0],
    'M': [1,1,1,1,1,1,1,1,1,1,1,1, 0,0,0,1,1,1,1,1,0,1,1,1,1,1]
}
df_minaret = pd.DataFrame(minaret_data, index=CANTONS)
print("Dataset — Swiss Minaret Vote (N=26 cantons):")
print(df_minaret.to_string())
print(f"\nX→M: {df_minaret[(df_minaret.X==1)&(df_minaret.M==1)].shape[0]} consistent / {df_minaret[df_minaret.X==1].shape[0]} X=1 cases, coverage {df_minaret[(df_minaret.X==1)&(df_minaret.M==1)].shape[0]}/{df_minaret[df_minaret.M==1].shape[0]}")


### Natural ⊥ candidates

The inner-Swiss mountain cantons (URI, OW, NW, AI, AR) had no significant Muslim community in 2009. The variable X ("new xenophobia") is arguably structurally inapplicable to these cantons — they voted for the ban through traditional religious conservatism, not through the social-contact mechanism X is designed to capture. This makes X=⊥ for these cantons a defensible protocol-driven inapplicability rather than MNAR.


In [ ]:
# CORAK analysis on clean data (no ⊥)
ctx_min = CorakContext(
    df_minaret.reset_index(drop=True),
    output_labels=['M'],
    input_labels=['A','L','S','T','X'],
    undef_value=-1,
    inc_score1=1.0
)
print("Row classification (clean data):")
print(ctx_min.get_row_classification()['row_class'].value_counts().to_string())
print("\nTier 1 prime implicants:")
for pi in ctx_min.get_prime_implicants():
    print(f"  {pi}")
print("\nConclusion: X→M and T→M are both prime implicants (disjunctive solution)")


In [ ]:
# Introduce ⊥ manually for the 5 inner-Swiss cantons (substantive coding)
df_min_coded = df_minaret.copy()
undef_cantons = ['UR', 'OW', 'NW', 'AI', 'AR']
df_min_coded.loc[undef_cantons, 'X'] = -1  # X=⊥ for mountain cantons

print(f"Cantons with X=⊥: {undef_cantons}")
print("\nCORA version (⊥→0):")
df_cora_min = df_min_coded.replace(-1, 0).reset_index(drop=True)
ctx_cora_min = CorakContext(df_cora_min, output_labels=['M'], input_labels=['A','L','S','T','X'],
                            undef_value=-1, inc_score1=1.0)
pis_cora = ctx_cora_min.get_prime_implicants()
x_standalone = [p for p in pis_cora if str(p)=='#X']
print(f"  Tier 1 PIs: {[str(p) for p in pis_cora]}")
print(f"  X→M standalone: {'YES ✓' if x_standalone else 'NO ✗ — fragmented into compound implicants'}")

print("\nCORAK version (⊥ as structural ⊥):")
ctx_kor_min = CorakContext(df_min_coded.reset_index(drop=True), output_labels=['M'],
                            input_labels=['A','L','S','T','X'], undef_value=-1, inc_score1=1.0)
pis_kor = ctx_kor_min.get_prime_implicants()
t2 = ctx_kor_min.get_conditional_prime_implicants()
print(f"  C1={ctx_kor_min.n_c1}  C2={ctx_kor_min.n_c2}  C3={ctx_kor_min.n_c3}")
print(f"  Tier 1 PIs: {[str(p) for p in pis_kor]}")
print(f"  Tier 2 conditionals: {len(t2)} (structural uncertainty flagged)")


### Monte Carlo simulation — varying ⊥ rate

We replicate the Section 5.3 simulation design on the Minaret dataset:
- **Mechanism A (C2 rows):** A random fraction r of X=1 cantons have X replaced by ⊥
- **Mechanism B (C3 rows):** A random fraction r of X=1,M=1 cantons have M replaced by ⊥

Recovery metric: X → M appears as a **standalone** prime implicant (`#X`).


In [ ]:
def x_recovers_standalone(pis):
    """True only if X=1 alone is a prime implicant (not embedded in conjunctions)."""
    for pi in pis:
        raw = pi.raw_implicant
        if (raw[4] == frozenset({1}) and
                all(raw[i] == frozenset({0,1}) for i in range(4))):
            return True
    return False

def perturb_minaret(base_df, rate, rng):
    df = base_df.copy()
    x1 = df[df['X'] == 1].index.tolist()
    n_a = round(len(x1) * rate)
    if n_a > 0:
        df.loc[rng.choice(x1, size=n_a, replace=False), 'X'] = -1
    x1m1 = df[(df['X']==1) & (df['M']==1)].index.tolist()
    n_b = round(len(x1m1) * rate)
    if n_b > 0:
        df.loc[rng.choice(x1m1, size=min(n_b, len(x1m1)), replace=False), 'M'] = -1
    return df

def run_one_minaret(df_p, undef_as_zero=False):
    df = df_p.replace(-1, 0) if undef_as_zero else df_p
    try:
        ctx = CorakContext(df.reset_index(drop=True), output_labels=['M'],
                           input_labels=['A','L','S','T','X'], undef_value=-1, inc_score1=1.0)
        pis = ctx.get_prime_implicants()
        recovered = x_recovers_standalone(pis)
        t2 = 0 if undef_as_zero else len(ctx.get_conditional_prime_implicants())
        return recovered, t2 > 0
    except Exception:
        return False, False

RATES = [0.00, 0.05, 0.10, 0.20, 0.30]
RUNS  = 200
rng   = np.random.default_rng(2024)

print(f"{'Rate':>6}  {'CORA':>8}  {'CORAK T1':>10}  {'Tier2':>8}  {'AvgC2':>6}  {'AvgC3':>6}")
print("-" * 55)
sim_results = []
for rate in RATES:
    cr=kr=t2=0; c2s=[]; c3s=[]
    for _ in range(RUNS):
        dp = perturb_minaret(df_minaret, rate, rng)
        c2s.append((dp['X']==-1).sum()); c3s.append((dp['M']==-1).sum())
        c,_  = run_one_minaret(dp, undef_as_zero=True)
        k,t  = run_one_minaret(dp, undef_as_zero=False)
        cr+=c; kr+=k; t2+=t
    row = dict(rate=f"{rate*100:.0f}%", cora=cr/RUNS, corak=kr/RUNS,
               tier2=t2/RUNS, avg_c2=np.mean(c2s), avg_c3=np.mean(c3s))
    sim_results.append(row)
    print(f"{rate*100:>5.0f}%  {cr/RUNS:>8.2f}  {kr/RUNS:>10.2f}  {t2/RUNS:>8.2f}  {np.mean(c2s):>6.1f}  {np.mean(c3s):>6.1f}")


### Cross-simulation comparison

| ⊥ Rate | CORA (multi-morbidity) | CORA (minaret) | CORAK T1 (MM) | CORAK T1 (min) | Tier 2 (MM) | Tier 2 (min) |
|--------|:---------------------:|:--------------:|:-------------:|:--------------:|:-----------:|:------------:|
| 0%  | 1.00 | 1.00 | 1.00 | 1.00 | 0.00 | 0.00 |
| 5%  | 0.10 | 0.00 | 1.00 | 1.00 | 0.51 | 1.00 |
| 10% | 0.06 | 0.00 | 1.00 | 1.00 | 0.76 | 1.00 |
| 20% | 0.01 | 0.00 | 1.00 | 0.98 | 0.96 | 1.00 |
| 30% | 0.00 | 0.00 | 1.00 | 0.98 | 0.99 | 1.00 |

**Key divergences:**
- CORA collapses **gradually** in multi-morbidity but **deterministically at 5%** in Minaret. The reason: the Minaret ON-set (19 X=1 cantons) is large enough that Mechanism B always places exactly 1 X=1,M=0 case even at 5%, while in multi-morbidity the smaller pool causes rounding to n_b=0 in some runs.
- CORAK Tier 1 is **always 1.00** in multi-morbidity (by construction) but **drops to 0.98 at 20-30%** in Minaret — a more honest test of evidence-thinning.  
- Tier 2 signal is **stronger in Minaret** (1.00 from 5%) than in multi-morbidity (0.51 at 5%), because both C2 and C3 rows are always present simultaneously in the Minaret runs.


### References

- Baumgartner, M., & Epple, R. (2014). A coincidence analysis of a causal chain: The Swiss minaret vote. *Sociological Methods & Research*, 43(2), 280–312.
- Thiem, A. (2015). Using Qualitative Comparative Analysis for identifying causal chains. *Sociological Methods & Research*, 44(4), 611–638. (QCA reanalysis of the minaret data)
